## 1. Import Libraries

In this section i import the libraries needed for the project:
- 'pathlib.Path': for file Path handling
- 'pandas': to load and manipulate dataset
- 'scikit-learn': for train/validation/test split
- 'collections.Counter': to check class balance
- 'transformers': to load the pretrained BERT model and tokenizer
- 'transformers.BertTokenizer': to convert text into tokens BERT can process
- 'transformers.BertForSequenceClassification': the pretrained BERT model with a classification head, used for fine-tuning on 3-classes task
- 'torch': for tensor operations and GPU support
- 'sklearn.metrics.accuracyscore: computes the overall accuracy of the model's predictions
- 'sklearn.metrics.f1_score': computes the F1 score, balancing precision and recall across classes
- 'transformers.TrainingArguments' : defines the hyperparameters and settings for the training process
- 'transformers.Trainer' : handles the full training and evaluation loop (forward pass, backpropagation, checkpoint saving, evaluation)

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

from transformers import BertTokenizer, BertForSequenceClassification
import torch

from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

In [ ]:
BASE_DIR = Path.cwd().parent

## 1.5 Functions and Class

Function:

- 'get_token_count': takes a single phrase as input, tokenizes it using the BERT tokenizer, and returns the number of tokens it was split into.
- 'compute_metrics' : takes the model's raw predictions (logits) and the true labels from an evaluation step, converts the logits into predicted class IDs, and returns a dictionary with accuracy and macro-averaged F1 score. Used by the 'Trainer' to evaluate performance at the end of each epoch.

Class:
- 'SkyGuardDataset':  a PyTorch 'Dataset' that wraps the tokenized encodings together with their numeric labels, so they can be fed directly into the Hugging Face 'Trainer'. It defines:
    - '__init__' : stores the tokenized encodings and the labels
    - '__len__' : returns the total number of samples
    - '__getitem__' : returns a single sample (tokenized input + its label) given an index

In [ ]:
# Functions

def get_token_count(text):
    tokens = tokenizer.encode(text) # tokenize the input text using BERT's tokenizer
    return len(tokens) # return how many tokens the phrase was split into


def compute_metrics(eval_pred):
    logits, labels = eval_pred # unpack model predictions and true labels
    predictions = logits.argmax(axis=-1) # convert logits to predicted class IDs
    acc = accuracy_score(labels, predictions) # overall accuracy
    f1 = f1_score(labels, predictions, average="macro") # F1 averaged equally across the 3 classes
    return {"accuracy": acc, "f1": f1}


# -------------------------------------------------------------------------------------------
# Class

class SkyGuardDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings  # tokenized input (input_ids, attention_mask)
        self.labels = labels        # numeric labels

    def __len__(self):
        return len(self.labels)  # total number of samples

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}  # get tokenized inputs for this sample
        item["labels"] = torch.tensor(self.labels[idx])                 # attach the corresponding label
        return item


## 2.  Import Dataset

I load the CSV file containing the labeled dataset (390 phrases split across the three classes: SOS, MAINTENANCE, SERVICE) and preview the first few rows to confirm it loaded correctly.

In [ ]:
df_Path = BASE_DIR / "data"/"raw"/"skyguard_dataset.csv"

df = pd.read_csv(df_Path)

df.head()

## 3. Dataset Quality Check

Before splitting the data, I check:
- the number of samples per class (to confirm balance)
- the presence of duplicate or near-duplicate rows
- any missing/empty values

In [ ]:
print("Class distribution:")
print(Counter(df["label"]))

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate texts only:", df["text"].duplicated().sum())

## 4. Train/Validation/Test split

I split the dataset into three parts:
- **Train**: used to fine-tune the model
- **Validation**: used to monitor the performance during training
- **Test**: held out completely, used for the final evaluation of the model

The split is stratified by label, so each subset keeps the same class proportions as the original dataset.

In [ ]:
train_df, temp_df = train_test_split( 
    df,
    test_size= 0.30,       # First split: Separate 70% of data for training (train_df) and 30% for a temporary set (temp_df)
    stratify= df['label'], # stratify keeps the class proportions identical to the original df
    random_state= 42       # random_state sets a fixed seed to guarantee reproducible splits
       )


val_df, test_df = train_test_split(  
    temp_df,               
    test_size= 0.50,            # Second split: Divide the temporary set equally (50/50) into validation (val_df) and test (test_df) sets      
    stratify= temp_df['label'], # This results in exactly 15% validation and 15% test of the total dataset (df)
    random_state= 42
)

print("Train size: ", len(train_df))
print("Validation size: ", len(val_df))
print("Test size: ", len(test_df))

print('\n')

print("Train class distribution: ", Counter(train_df['label']))
print("Validation class distribution: ", Counter(val_df['label']))
print("Test class distribution: ", Counter(test_df['label']))

## 5. Save splits to disk

I save the train, validation, and test sets as separate CSV files in `data/processed/`, so they can be reused without recomputing the split.

In [ ]:
train_df.to_csv(BASE_DIR / "data" / "processed" / "train.csv", index= False)
val_df.to_csv(BASE_DIR / "data" / "processed" / "val.csv", index= False)
test_df.to_csv(BASE_DIR / "data" / "processed" / "test.csv", index = False)

print("Files saved successfully in 'data/processed/'")

## 6. Load Pretrained Model and Tokenizer

I load 'bert-base-uncased' from Hugging Face, along with its tokenizer.

Since  I have 3 classes (SOS, MAINTENANCE; SERVICE), I configure the model with 'num_label = 3' and a mapping between labels and their numeric IDs.

In [ ]:
label2id = {"SOS" : 0, "MAINTENANCE" : 1, "SERVICE" : 2} # maps each class name to a numeric ID
id2label = {0 : "SOS", 1 : "MAINTENANCE", 2 : "SERVICE"} # reverse mapping, numeric ID back to class name

model_name = "bert-base-uncased" # pretrained model to fine-tune

tokenizer = BertTokenizer.from_pretrained(model_name)  # loads the tokenizer matching bert-base-uncased
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 3, # 3 output classes: SOS, MAINTENANCE, SERVICE
    label2id = label2id, # required by Hugging Face to map labels to IDs
    id2label = id2label  # required by Hugging Face to map IDs back to labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # use GPU if available, otherwise fall back to CPU

# Explicit equivalent of the inline 'if' statement above for clarity:
# if torch.cuda.is_available():
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")

model.to(device) # move model weights to the selected device

print("Model loaded on: ", device)

## 7. Tokenization

Before tokenizing the full dataset, I check the token lenght distribution of the phrases to choose an appropriate 'max_length' value (avoiding unnecessary padding or truncation)

Then I tokenize the train, validation and test sets using the BERT tokenizer, converting each phrase into input IDs and attention masks the model can process.

In [ ]:
token_lengths = df["text"].apply(get_token_count)  # apply the 'get_token_count' function to every phrase in the dataset


print("Max token length:", token_lengths.max())
print("Mean token length:", token_lengths.mean())
print("95th percentile:", token_lengths.quantile(0.95))

Based on the token length analysis above (max 38, mean ~22, 95th percentile 28), I set 'max_length=50': a round number that comfortably covers the longest phrase in the dataset while keeping tokenization lightweight.

In [ ]:
MAX_LENGTH = 50 # chosen based on the token length analysis before

train_encodings = tokenizer(
    list(train_df["text"]),
    truncation = True,
    padding = "max_length",
    max_length = MAX_LENGTH,
    return_tensors = "pt"
)

val_encodings = tokenizer(
    list(val_df["text"]),
    truncation = True,
    padding = "max_length",
    max_length = MAX_LENGTH,
    return_tensors = "pt"
)

test_encodings = tokenizer(
    list(test_df["text"]),
    truncation = True,
    padding = "max_length",
    max_length = MAX_LENGTH,
    return_tensors = "pt"
)


print("Train encodings shape:", train_encodings["input_ids"].shape)
print("Validation encodings shape:", val_encodings["input_ids"].shape)
print("Test encodings shape:", test_encodings["input_ids"].shape)

## 8. Labels and Pytorch

I convert the text labels (SOS, MAINTENANCE, SERVICE) into numeric IDs using 'label2id', then I build a PyTorch 'Dataset' class that combines the tokenized encodings with their corresponding labels, in the format expected by the Hugging Face 'Trainer'.

In [ ]:
train_labels = [label2id[label] for label in train_df['label']] # convert text labels to numeric IDs
val_labels = [label2id[label] for label in val_df['label']]
test_labels = [label2id[label] for label in test_df['label']]

# Explicit equivalent

# train_labels = [] # 1. Initialize an empty list
# for label in train_df['label'] :  # 2. Loop through each label in the DataFrame column
#     id_associated = label2id[label] # 3. Look up the ID in your dictionary
#     train_labels.append(id_associated) # 4. Append the ID to the final list

# val_labels = []
# for label in val_df['label']:
#     id_associated = label2id[label]
#     val_labels.append(id_associated)

# test_labels = []
# for label in test_df['label']:
#     id_associated = label2id[label]
#     test_labels.append(id_associated)

In [ ]:
train_dataset = SkyGuardDataset(train_encodings, train_labels)
val_dataset = SkyGuardDataset(val_encodings,  val_labels)
test_dataset = SkyGuardDataset(test_encodings, test_labels)

print('Train Dataset size: ', len(train_dataset))
print('Val Dataset size: ', len(val_dataset))
print('Test Dataset size: ', len(test_dataset))

## 9. Training Configuration

I define the training hyperparameters (epochs, batch size, learning rate, evaluation strategy) using 'TrainingArguments', and I create a 'Trainer' object that handles the training and validation loop, along with a function to compute accuracy and F1 score at each evaluation step.

In [ ]:
training_args = TrainingArguments(
    output_dir = str(BASE_DIR / "models" / "training_logs"),  # where checkpoints and logs are saved
    num_train_epochs = 5, # small dataset fewe epochs are  enough
    per_device_train_batch_size = 16, # number of training samples processed together in one forward/backward pass, per GPU/CPU
    per_device_eval_batch_size = 16, # same concept, but used during evaluation (no backpropagation, so it could technically be higher without memory issues)
    learning_rate = 2e-5,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "f1",
    save_total_limit = 2  # keep only the 2 most recent checkpoints, delete older ones automatically
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset= train_dataset,
    eval_dataset= val_dataset,
    compute_metrics= compute_metrics
)

## 10. Training

In [ ]:
trainer.train()  # starts the fine-tuning process